## Model Training

In [113]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline
# Modelling
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

In [114]:
df = pd.read_csv('data/stud.csv')
df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [115]:
X = df.drop(['math_score'],axis=1)
y = df['math_score']

In [116]:
X

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,74
1,female,group C,some college,standard,completed,90,88
2,female,group B,master's degree,standard,none,95,93
3,male,group A,associate's degree,free/reduced,none,57,44
4,male,group C,some college,standard,none,78,75
...,...,...,...,...,...,...,...
995,female,group E,master's degree,standard,completed,99,95
996,male,group C,high school,free/reduced,none,55,55
997,female,group C,high school,free/reduced,completed,71,65
998,female,group D,some college,standard,completed,78,77


In [117]:
y

0      72
1      69
2      90
3      47
4      76
       ..
995    88
996    62
997    59
998    68
999    77
Name: math_score, Length: 1000, dtype: int64

In [118]:
num_features = X.select_dtypes(exclude='object').columns
cat_features = X.select_dtypes(include='object').columns

print('Numeric Features: ', num_features)
print('Categorical Features: ', cat_features)

Numeric Features:  Index(['reading_score', 'writing_score'], dtype='object')
Categorical Features:  Index(['gender', 'race_ethnicity', 'parental_level_of_education', 'lunch',
       'test_preparation_course'],
      dtype='object')


In [119]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [120]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
num_transformer = StandardScaler()
cat_transformer = OneHotEncoder(drop='first')

preprocessor = ColumnTransformer(
    [
        ('StandardScaler', num_transformer, num_features),
        ('OneHotEncoder', cat_transformer, cat_features)
    ]
)

In [121]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [122]:
def eval_model(true, pred):
    mae = mean_absolute_error(true, pred)
    mse = mean_squared_error(true, pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(true, pred)
    adj_r2 = 1 - (1-r2)*(len(true)-1)/(len(true)-X.shape[1]-1)
    return mae, mse, rmse, r2, adj_r2

In [123]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'KNN': KNeighborsRegressor(),
    'Decision Tree': DecisionTreeRegressor(),
    'Random Forest': RandomForestRegressor(),
    'AdaBoost': AdaBoostRegressor(),
    'SVM': SVR(),
    'CatBoost': CatBoostRegressor(verbose=0),
    'XGBoost': XGBRegressor()
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_mae, train_mse, train_rmse, train_r2, train_adj_r2 = eval_model(y_train, y_train_pred)
    test_mae, test_mse, test_rmse, test_r2, test_adj_r2 = eval_model(y_test, y_test_pred)

    results.append({
        'Model': name,
        'accuracy': test_r2,
    })

    print(f'=============================================== {name} ===========================================')
    print('\nModel Performance for Trainning Set')
    print('Mean Absolute Error:', train_mae)
    print('Mean Squared Error:', train_mse)
    print('Root Mean Squared Error:', train_rmse)
    print('R2 Score:', train_r2)
    print('Adjusted R2 Score:', train_adj_r2)
    print('---------------------------------------------------------------------------------------------------')
    print('\nModel Performance for Test Set')
    print('Mean Absolute Error:', test_mae)
    print('Mean Squared Error:', test_mse)
    print('Root Mean Squared Error:', test_rmse)
    print('R2 Score:', test_r2)
    print('Adjusted R2 Score:', test_adj_r2)
    print('====================================================================================================')
    print('\n')

=============================================== Linear Regression ===========================================

Model Performance for Trainning Set
Mean Absolute Error: 4.266711846071957
Mean Squared Error: 28.33487038064859
Root Mean Squared Error: 5.323050852720514
R2 Score: 0.8743172040139593
Adjusted R2 Score: 0.8732063712211534
---------------------------------------------------------------------------------------------------

Model Performance for Test Set
Mean Absolute Error: 4.214763142474852
Mean Squared Error: 29.0951698667155
Root Mean Squared Error: 5.393993869732844
R2 Score: 0.8804332983749564
Adjusted R2 Score: 0.8760740957115434


=============================================== Ridge ===========================================

Model Performance for Trainning Set
Mean Absolute Error: 4.26500882127927
Mean Squared Error: 28.33963485875214
Root Mean Squared Error: 5.3234983665586055
R2 Score: 0.8742960705864397
Adjusted R2 Score: 0.8731850510082895
------------------------

In [124]:
results = pd.DataFrame(results)
results

,Model,accuracy
0,Linear Regression,0.880433
1,Ridge,0.880450
2,Lasso,0.825447
3,KNN,0.777262
4,Decision Tree,0.752854
5,Random Forest,0.848299
6,AdaBoost,0.854966
7,SVM,0.713544
8,CatBoost,0.850263
9,XGBoost,0.820924


In [125]:
results = results.sort_values(by='accuracy', ascending=False)
results

,Model,accuracy
1,Ridge,0.880450
0,Linear Regression,0.880433
6,AdaBoost,0.854966
8,CatBoost,0.850263
5,Random Forest,0.848299
2,Lasso,0.825447
9,XGBoost,0.820924
3,KNN,0.777262
4,Decision Tree,0.752854
7,SVM,0.713544


In [126]:
results = results.head()
results

,Model,accuracy
1,Ridge,0.880450
0,Linear Regression,0.880433
6,AdaBoost,0.854966
8,CatBoost,0.850263
5,Random Forest,0.848299


# Hyperparameter Tuning

In [127]:
rf_params = {
    'criterion': ['squared_error', 'friedman_mse', 'poisson'],
    'max_depth': [None, 1, 5, 10, 15, 20],
    'min_samples_split': [2, 8, 15, 20],
    'n_estimators': [10, 50, 100, 200, 500],
    'max_features': [None, 'sqrt', 'log2']
}

ab_params = {
    'n_estimators': [10, 50, 100, 200, 500],
    'learning_rate': [0.1, 0.5, 1.0, 1.5, 2.0],
    'loss': ['linear', 'square', 'exponential'],
}

In [128]:
from sklearn.model_selection import GridSearchCV

grid_models = [
    ('Random Forest', RandomForestRegressor(), rf_params),
    ('AdaBoost', AdaBoostRegressor(), ab_params),
]

results = []
for name, model, params in grid_models:
    grid = GridSearchCV(estimator=model, param_grid=params, cv=3, scoring='accuracy', n_jobs=-1, verbose=2)
    grid.fit(X_train, y_train)

    y_train_pred = grid.predict(X_train)
    y_test_pred = grid.predict(X_test)

    train_mae, train_mse, train_rmse, train_r2, train_adj_r2 = eval_model(y_train, y_train_pred)
    test_mae, test_mse, test_rmse, test_r2, test_adj_r2 = eval_model(y_test, y_test_pred)

    results.append({
        'Model': model,
        'Params': grid.best_params_,
        'accuracy': test_r2,
    })

    print(f'=============================================== {name} ===========================================')
    print('\nModel Performance for Trainning Set')
    print('Mean Absolute Error:', train_mae)
    print('Mean Squared Error:', train_mse)
    print('Root Mean Squared Error:', train_rmse)
    print('R2 Score:', train_r2)
    print('Adjusted R2 Score:', train_adj_r2)
    print('---------------------------------------------------------------------------------------------------')
    print('\nModel Performance for Test Set')
    print('Mean Absolute Error:', test_mae)
    print('Mean Squared Error:', test_mse)
    print('Root Mean Squared Error:', test_rmse)
    print('R2 Score:', test_r2)
    print('Adjusted R2 Score:', test_adj_r2)
    print('====================================================================================================')
    print('\n')

Fitting 3 folds for each of 1080 candidates, totalling 3240 fits
=============================================== Random Forest ===========================================

Model Performance for Trainning Set
Mean Absolute Error: 2.013125
Mean Squared Error: 7.332313194444445
Root Mean Squared Error: 2.707824439369075
R2 Score: 0.967476624705067
Adjusted R2 Score: 0.9671891706304906
---------------------------------------------------------------------------------------------------

Model Performance for Test Set
Mean Absolute Error: 4.77275
Mean Squared Error: 39.2619125
Root Mean Squared Error: 6.265932691946188
R2 Score: 0.8386530342107945
Adjusted R2 Score: 0.8327705927497298


Fitting 3 folds for each of 75 candidates, totalling 225 fits
=============================================== AdaBoost ===========================================

Model Performance for Trainning Set
Mean Absolute Error: 5.762213568994932
Mean Squared Error: 50.2657137199127
Root Mean Squared Error: 7.08983171

In [129]:
results = pd.DataFrame(results)
results.sort_values(by='accuracy', ascending=False)

,Model,Params,accuracy
0,RandomForestRegressor(),"{'criterion': 'squared_error', 'max_depth': No...",0.838653
1,AdaBoostRegressor(),"{'learning_rate': 0.1, 'loss': 'linear', 'n_es...",0.776529


In [130]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

num_features = X.select_dtypes(exclude='object').columns
cat_features = X.select_dtypes(include='object').columns

preprocessor = ColumnTransformer(
    [
        ('StandardScaler', StandardScaler(), num_features),
        ('OneHotEncoder', OneHotEncoder(), cat_features)
    ], remainder='passthrough'
)

In [131]:
from sklearn.pipeline import Pipeline

pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

pipe.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('StandardScaler', ...), ('OneHotEncoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
